### environment

In [ ]:
%pip install -U spacy
%pip install --upgrade pip
%pip install pip setuptools wheel sklearn
!python -m spacy download en_core_web_sm 

In [1]:
import os, rich, json
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import spacy
from spacy.tokens import DocBin, Doc
from label_models import Label

In [9]:
import pandas as pd

In [ ]:
# spacy.prefer_gpu()

In [2]:
nlp = spacy.load("en_core_web_sm") 
doc_bin = DocBin()

/home/fit-ls36/linux-home/repos-temp/CarP-NER-spacy-CNN/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### processing

In [3]:
with open(f'{os.getcwd()}/labels_en.json','r') as f:
    dataset = json.loads(f.read())
    train, test = train_test_split(dataset, test_size=0.2, random_state=42)
    
#    with open('train.json','w') as tr:
#        tr.write(json.dumps(train))
#    with open('test.json','w') as te:
#        te.write(json.dumps(test))

In [4]:
def remove_whitespace(content, labels):
    leading = len(content) - len(content.lstrip())
    content_no_ws = content.strip()
    new_labels = []

    for label in labels:
        start, end, label_type, value = label.start, label.end, label.label, label.value
        label_leading = len(value) - len(value.lstrip())

        start += label_leading - leading
        end = start + len(value.strip()) - leading
        value = value.strip()

        if start >= 0 and end <= len(content_no_ws):
            if value == value.strip():
                new_labels.append(Label(start,end,label_type,value))
            else:
                print(f'unresolved whitespace: {label.value}')
        else: print(f'invalid indices {start}, {end}')

    return content_no_ws, new_labels

In [5]:
def split_doc(text, labels, MAX_TOKENS=500, MIN_TOKENS = 10):
    sequences = []
    current_seq = []
    current_offset = 0 
    '''
    split text into sequences short enough (MAX_TOKENS) to be used in spacy's model training
    MIN_TOKENS: to create reasonably long sequences in case of last few remaining tokens
    '''
    text_doc = nlp(text)
    for token in text_doc:
        current_seq.append(token)
        if len(current_seq) >= MAX_TOKENS:
            seq_text = "".join(     # join all tokens in sequence into a doc again
                [t.text_with_ws for t in current_seq])      # whitespace included in individual tokens to preserve label offsets
            seq_labels = [ 
                Label(
                    label['start'] - current_offset, # adjust label indexes to preserve original data
                    label['end'] - current_offset,
                    label['label'],
                    label['value']
                )
                for label in labels # add labels only in the current chunk
                if label["start"] >= current_offset and label["end"] <= current_offset + len(seq_text)
            ]

            sequences.append((seq_text, seq_labels)) 
            current_offset += len(seq_text)
            current_seq = [] # reset for next sequence' tokens
    
    if current_seq: # handle remaining tokens
        seq_text = "".join([t.text_with_ws for t in current_seq])
        seq_labels = [
            Label(
                label['start'] - current_offset, 
                label['end'] - current_offset,
                label['label'],
                label['value']
            )
            for label in labels 
            if label["start"] >= current_offset and label["end"] <= current_offset + len(seq_text)
        ]
        if len(current_seq) > MIN_TOKENS or len(seq_labels) > 0:
            return (seq_text, seq_labels)


In [ ]:
texts = []
labels_lists = []

for d in train:
    t = split_doc(d['content'], d['labels'])
    if t != None:
        text, labels = remove_whitespace(t[0],t[1])
        texts.append(text)
        labels_lists.append(labels)

print(len(texts))


1914


In [28]:
objs = []
for i in range(0,1913):
    if labels_lists[i] == []: 
        continue
    else:
        text, labels = texts[i], labels_lists[i]
        labels = [label.to_dict() for label in labels]
        objs.append({"text":text,"labels":labels})

In [30]:
df = pd.DataFrame(objs)
df['labels']

0      [{'start': 47, 'end': 62, 'label': 'CAPABILITY...
1      [{'start': 9, 'end': 19, 'label': 'JOB_TITLE',...
2      [{'start': 1025, 'end': 1040, 'label': 'JOB_TI...
3      [{'start': 569, 'end': 598, 'label': 'JOB_TITL...
4      [{'start': 0, 'end': 24, 'label': 'JOB_TITLE',...
                             ...                        
953    [{'start': 3, 'end': 11, 'label': 'CAPABILITY'...
954    [{'start': 2323, 'end': 2336, 'label': 'SOFT_S...
955    [{'start': 1360, 'end': 1381, 'label': 'SOFT_S...
956    [{'start': 351, 'end': 386, 'label': 'JOB_TITL...
957    [{'start': 84, 'end': 112, 'label': 'CAPABILIT...
Name: labels, Length: 958, dtype: object

In [31]:
with open('resumes_train.json','w') as j:
    j.write(json.dumps(objs))

In [ ]:
docs = list(nlp.pipe((texts[0] for texts[0] in texts), batch_size=6)) # reinit docs in batches from cleaned texts

In [ ]:
empty_span_count, overlap_count, ent_count = 0,0,0

for doc,labels in tqdm(zip(docs,labels_lists)):
    ents = []
    for l in labels:
        start, end, ltype, value = l.start, l.end, l.label, l.value
        if start > end: # invalid indices
            continue
        span = doc.char_span(start,end,ltype,alignment_mode='expand')
        if span is None:
            empty_span_count += 1
            continue
        if any(
                span.start < existing_span.end and 
                span.end > existing_span.start 
            for existing_span in ents):
            overlap_count += 1 #check overlapping spand caused by alignment_mode in some cases
            continue
        ents.append(span)
        ent_count += len(ents)
    if len(ents) > 0: # filter out (mostly short) docs with no valid entities
        doc.set_ents(ents)  
        doc_bin.add(doc)
doc_bin.to_disk("training.spacy")
print(f'final entities: {ent_count}\noverlaps: {overlap_count}\nempty spans:{empty_span_count}')

In [ ]:
doc_bin = DocBin().from_disk("test.spacy")
cleaned = DocBin()

ents_len = 0
for doc in doc_bin.get_docs(nlp.vocab):
    ents = [ent for ent in doc.ents if ent.text == ent.text.strip()]
    doc.set_ents(ents)
    ents_len += len(ents)
    cleaned.add(doc)

cleaned.to_disk('eval.spacy')
print(ents_len)
